# S1 Preprocessing and Descriptor Selection

This notebook implements the preprocessing and descriptor-selection workflow used to prepare molecular data for QSPR modeling of **Log_S1**.

The workflow includes:

* loading and, if necessary, combining molecular descriptor tables from multiple Excel files;
* removing constant and non-modeling columns;
* reducing descriptor redundancy using a **Pearson correlation filter** (`|r| > 0.95`);
* defining a fixed **external test set** containing only metal–DNA base complexes;
* keeping isolated bimetallic clusters and individual DNA bases in the training set;
* preparing descriptor chunks for **GA-MLR feature selection** using DTC Lab Tools;
* parsing GA-MLR output files and calculating descriptor selection frequencies;
* performing **consensus descriptor selection** based on repeated occurrence across GA-MLR equations;
* generating the reduced descriptor dataset used for subsequent QSPR modeling;

The external test molecule IDs are stored explicitly to ensure that the same train/test partition is used in subsequent modeling workflows.

### Workflow overview

**Raw molecular descriptors**
→ constant-feature removal
→ correlation filtering
→ fixed external test selection
→ GA-MLR descriptor screening
→ frequency-based consensus selection
→ reduced descriptor set

The notebook is intended primarily for **data preprocessing and descriptor selection**. Model training, hyperparameter optimization, validation, applicability-domain analysis, and model interpretation are performed in the subsequent QSPR modeling workflow.


In [1]:
# Import libraries for data handling, numerical operations,
# file management, and data visualization.
import pandas as pd
import numpy as np
import seaborn as sns
import os
import glob
import matplotlib.pyplot as plt

In [2]:
# Check the current working directory.
# Uncomment this line if the location of the input files needs to be verified.
# os.getcwd()

In [3]:
# Optionally change the working directory to the folder containing
# the input descriptor and target files.
#os.chdir('directory')

In [4]:
# Verify that the working directory was changed successfully.
# os.getcwd()

In [5]:
# OPTIONAL: use this block when molecular descriptors are stored
# across multiple Excel workbooks.
# Find all .xlsx files in the current working directory.
# xl_files = glob.glob('*.xlsx')
# xl_files

In [6]:
# Inspect the sheet structure of the first Excel workbook
# before combining descriptor tables.
# sample_xl =pd.ExcelFile(xl_files[0])
# sample_xl.sheet_names

In [8]:
# Read all sheets from all detected Excel workbooks and combine them
# into a single descriptor table.
# combined_list = []
#
# for xl_file in xl_files:
#     xl_file_obj = pd.ExcelFile(xl_file)
#     for sheet_name in xl_file_obj.sheet_names:
#         data = pd.read_excel(xl_file_obj, sheet_name=sheet_name)
#         combined_list.append(data)
#
# # Combine all DataFrames by columns (to the right).
# combined = pd.concat(combined_list, axis=1)


In [9]:
# Inspect the combined descriptor table before saving.
# combined

In [10]:
# Save the combined molecular descriptor table for subsequent preprocessing.
# combined.to_excel('All_descriptors_combined.xlsx', index=False)

In [11]:
# Load the complete molecular descriptor matrix.
# Start from this block if all descriptors have already been combined
# into a single Excel file.
data = pd.read_excel('All_descriptors_combined.xlsx')

In [12]:
# Inspect the raw descriptor matrix and verify that it was loaded correctly.
data

,SMILES,CASRN,EXTERNALID,N,NAME,ARTICLEID,PUBMEDID,PAGE,TABLE,ERROR,...,Se1Cu1Au2s:(OEstate),Se1Cu1Cu2s:(OEstate),Se1N3Cu2ss:(OEstate),SssCu:(OEstate),Se1O2Cu2ss:(OEstate),Se1Cu2Ag1s:(OEstate),Se1Cu2Au1s:(OEstate),Se1C1C3s:(OEstate),Se1C3C3ss:(OEstate),SsCH3:(OEstate)
0,[Ag][Ag],NaN,NaN,NaN,NaN,-,-,-,-,NaN,...,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0,0.0000,0.0
1,[Ag][Au],NaN,NaN,NaN,NaN,-,-,-,-,NaN,...,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0,0.0000,0.0
2,[Cu][Ag],NaN,NaN,NaN,NaN,-,-,-,-,NaN,...,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0,0.0000,0.0
3,[Au][Au],NaN,NaN,NaN,NaN,-,-,-,-,NaN,...,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0,0.0000,0.0
4,[Cu][Au],NaN,NaN,NaN,NaN,-,-,-,-,NaN,...,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0,0.0000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,NC1NC2NCN([Cu][Ag])C2C(O)N1,NaN,NaN,NaN,NaN,-,-,-,-,NaN,...,0.0,0.0,0.7628,-0.6877,0.000,0.13190,0.000000,0.0,0.2681,0.0
96,NC1NC2NCNC2C(N1)O[Cu][Ag],NaN,NaN,NaN,NaN,-,-,-,-,NaN,...,0.0,0.0,0.0000,-0.8216,2.426,0.04336,0.000000,0.0,0.3630,0.0
97,NC1NC(O)C2NCNC2N1[Cu][Au],NaN,NaN,NaN,NaN,-,-,-,-,NaN,...,0.0,0.0,0.7026,-0.7749,0.000,0.00000,0.036110,0.0,0.1902,0.0
98,NC1NC2NCN([Cu][Au])C2C(O)N1,NaN,NaN,NaN,NaN,-,-,-,-,NaN,...,0.0,0.0,0.7689,-0.6755,0.000,0.00000,0.084980,0.0,0.2711,0.0


In [12]:
# Remove constant descriptors.
# Features containing only one unique value provide no information
# for distinguishing molecules and are therefore excluded.
n_unique = data.nunique()
df = data.loc[:, n_unique > 1]

In [13]:
# Inspect the dataset after removing constant descriptors.
df

,SMILES,ALogPS_logP,ALogPS_logS,MW:(alvaDesc),AMW:(alvaDesc),Sv:(alvaDesc),Se:(alvaDesc),Sp:(alvaDesc),Si:(alvaDesc),Mv:(alvaDesc),...,Se1Cu1Au2s:(OEstate),Se1Cu1Cu2s:(OEstate),Se1N3Cu2ss:(OEstate),SssCu:(OEstate),Se1O2Cu2ss:(OEstate),Se1Cu2Ag1s:(OEstate),Se1Cu2Au1s:(OEstate),Se1C1C3s:(OEstate),Se1C3C3ss:(OEstate),SsCH3:(OEstate)
0,[Ag][Ag],-1.30,-0.15,215.7,107.90,2.071,1.331,8.182,1.346,1.0360,...,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0,0.0000,0.0
1,[Ag][Au],-1.30,-0.43,304.8,152.40,1.967,0.000,7.386,1.492,0.9833,...,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0,0.0000,0.0
2,[Cu][Ag],-1.30,-0.08,171.4,85.71,1.594,1.385,7.557,1.359,0.7969,...,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0,0.0000,0.0
3,[Au][Au],-1.30,-0.61,393.9,197.00,1.862,0.000,6.591,1.639,0.9310,...,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0,0.0000,0.0
4,[Cu][Au],-1.30,-0.27,260.5,130.30,1.489,0.000,6.761,1.505,0.7447,...,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0,0.0000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,NC1NC2NCN([Cu][Ag])C2C(O)N1,-2.43,0.18,329.6,13.19,14.260,24.810,20.700,28.510,0.5704,...,0.0,0.0,0.7628,-0.6877,0.000,0.13190,0.000000,0.0,0.2681,0.0
96,NC1NC2NCNC2C(N1)O[Cu][Ag],-2.02,-0.25,329.6,13.19,14.260,24.810,20.700,28.510,0.5704,...,0.0,0.0,0.0000,-0.8216,2.426,0.04336,0.000000,0.0,0.3630,0.0
97,NC1NC(O)C2NCNC2N1[Cu][Au],-2.37,-0.43,418.7,16.75,14.150,0.000,19.910,28.660,0.5662,...,0.0,0.0,0.7026,-0.7749,0.000,0.00000,0.036110,0.0,0.1902,0.0
98,NC1NC2NCN([Cu][Au])C2C(O)N1,-2.35,-0.43,418.7,16.75,14.150,0.000,19.910,28.660,0.5662,...,0.0,0.0,0.7689,-0.6755,0.000,0.00000,0.084980,0.0,0.2711,0.0


In [14]:
# Remove the SMILES column because it is used only as a molecular
# identifier and is not included as a numerical model descriptor.
df = df.drop(columns=['SMILES'])

In [15]:
# Inspect the descriptor matrix after removing the SMILES column.
df

,ALogPS_logP,ALogPS_logS,MW:(alvaDesc),AMW:(alvaDesc),Sv:(alvaDesc),Se:(alvaDesc),Sp:(alvaDesc),Si:(alvaDesc),Mv:(alvaDesc),Me:(alvaDesc),...,Se1Cu1Au2s:(OEstate),Se1Cu1Cu2s:(OEstate),Se1N3Cu2ss:(OEstate),SssCu:(OEstate),Se1O2Cu2ss:(OEstate),Se1Cu2Ag1s:(OEstate),Se1Cu2Au1s:(OEstate),Se1C1C3s:(OEstate),Se1C3C3ss:(OEstate),SsCH3:(OEstate)
0,-1.30,-0.15,215.7,107.90,2.071,1.331,8.182,1.346,1.0360,0.6655,...,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0,0.0000,0.0
1,-1.30,-0.43,304.8,152.40,1.967,0.000,7.386,1.492,0.9833,0.0000,...,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0,0.0000,0.0
2,-1.30,-0.08,171.4,85.71,1.594,1.385,7.557,1.359,0.7969,0.6927,...,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0,0.0000,0.0
3,-1.30,-0.61,393.9,197.00,1.862,0.000,6.591,1.639,0.9310,0.0000,...,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0,0.0000,0.0
4,-1.30,-0.27,260.5,130.30,1.489,0.000,6.761,1.505,0.7447,0.0000,...,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0,0.0000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,-2.43,0.18,329.6,13.19,14.260,24.810,20.700,28.510,0.5704,0.9926,...,0.0,0.0,0.7628,-0.6877,0.000,0.13190,0.000000,0.0,0.2681,0.0
96,-2.02,-0.25,329.6,13.19,14.260,24.810,20.700,28.510,0.5704,0.9926,...,0.0,0.0,0.0000,-0.8216,2.426,0.04336,0.000000,0.0,0.3630,0.0
97,-2.37,-0.43,418.7,16.75,14.150,0.000,19.910,28.660,0.5662,0.0000,...,0.0,0.0,0.7026,-0.7749,0.000,0.00000,0.036110,0.0,0.1902,0.0
98,-2.35,-0.43,418.7,16.75,14.150,0.000,19.910,28.660,0.5662,0.0000,...,0.0,0.0,0.7689,-0.6755,0.000,0.00000,0.084980,0.0,0.2711,0.0


In [16]:
# Remove highly correlated descriptors using pairwise Pearson correlation.
# For each descriptor pair with |r| > 0.95, one descriptor is removed
# to reduce redundancy and dimensionality of the feature space.
#
# IMPORTANT:
# For a strictly leakage-free modeling workflow, the correlation filter
# should be fitted using the training set only and then applied unchanged
# to the external test set.
corr_matrix = df.corr().abs()

# Select the upper triangle (excluding duplicates and the diagonal)
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find features with correlation above the threshold
threshold = 0.95
to_drop = [column for column in upper.columns if any(upper[column] > threshold)]

# Drop those features
df = df.drop(columns=to_drop)

print(f' {len(to_drop)} features with correlation > {threshold} were removed.')

 2147 features with correlation > 0.95 were removed.


In [17]:
# Inspect the descriptor matrix after correlation filtering.
df

,ALogPS_logP,ALogPS_logS,MW:(alvaDesc),AMW:(alvaDesc),Sv:(alvaDesc),Se:(alvaDesc),Sp:(alvaDesc),Mv:(alvaDesc),Mp:(alvaDesc),nTA:(alvaDesc),...,Se1O2Au2ss:(OEstate),Se1Ag1Au2s:(OEstate),Se1Cu1Au2s:(OEstate),Se1Cu1Cu2s:(OEstate),Se1N3Cu2ss:(OEstate),SssCu:(OEstate),Se1O2Cu2ss:(OEstate),Se1Cu2Ag1s:(OEstate),Se1Cu2Au1s:(OEstate),Se1C3C3ss:(OEstate)
0,-1.30,-0.15,215.7,107.90,2.071,1.331,8.182,1.0360,4.0910,2,...,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000
1,-1.30,-0.43,304.8,152.40,1.967,0.000,7.386,0.9833,3.6930,2,...,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000
2,-1.30,-0.08,171.4,85.71,1.594,1.385,7.557,0.7969,3.7780,2,...,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000
3,-1.30,-0.61,393.9,197.00,1.862,0.000,6.591,0.9310,3.2960,2,...,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000
4,-1.30,-0.27,260.5,130.30,1.489,0.000,6.761,0.7447,3.3810,2,...,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,-2.43,0.18,329.6,13.19,14.260,24.810,20.700,0.5704,0.8282,3,...,0.0,0.0,0.0,0.0,0.7628,-0.6877,0.000,0.13190,0.000000,0.2681
96,-2.02,-0.25,329.6,13.19,14.260,24.810,20.700,0.5704,0.8282,2,...,0.0,0.0,0.0,0.0,0.0000,-0.8216,2.426,0.04336,0.000000,0.3630
97,-2.37,-0.43,418.7,16.75,14.150,0.000,19.910,0.5662,0.7964,3,...,0.0,0.0,0.0,0.0,0.7026,-0.7749,0.000,0.00000,0.036110,0.1902
98,-2.35,-0.43,418.7,16.75,14.150,0.000,19.910,0.5662,0.7964,3,...,0.0,0.0,0.0,0.0,0.7689,-0.6755,0.000,0.00000,0.084980,0.2711


In [18]:
# Assign sequential molecule identifiers starting from 1.
# The index is named "SrNo" to match the input format required
# by DTC Lab Tools for GA-MLR descriptor selection.
df.index = range(1, len(df) + 1)

df.index.name = 'SrNo'

In [19]:
# Inspect the dataset and verify the molecule identifiers.
df

,ALogPS_logP,ALogPS_logS,MW:(alvaDesc),AMW:(alvaDesc),Sv:(alvaDesc),Se:(alvaDesc),Sp:(alvaDesc),Mv:(alvaDesc),Mp:(alvaDesc),nTA:(alvaDesc),...,Se1O2Au2ss:(OEstate),Se1Ag1Au2s:(OEstate),Se1Cu1Au2s:(OEstate),Se1Cu1Cu2s:(OEstate),Se1N3Cu2ss:(OEstate),SssCu:(OEstate),Se1O2Cu2ss:(OEstate),Se1Cu2Ag1s:(OEstate),Se1Cu2Au1s:(OEstate),Se1C3C3ss:(OEstate)
SrNo,,,,,,,,,,,,,,,,,,,,,
1,-1.30,-0.15,215.7,107.90,2.071,1.331,8.182,1.0360,4.0910,2,...,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000
2,-1.30,-0.43,304.8,152.40,1.967,0.000,7.386,0.9833,3.6930,2,...,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000
3,-1.30,-0.08,171.4,85.71,1.594,1.385,7.557,0.7969,3.7780,2,...,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000
4,-1.30,-0.61,393.9,197.00,1.862,0.000,6.591,0.9310,3.2960,2,...,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000
5,-1.30,-0.27,260.5,130.30,1.489,0.000,6.761,0.7447,3.3810,2,...,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,-2.43,0.18,329.6,13.19,14.260,24.810,20.700,0.5704,0.8282,3,...,0.0,0.0,0.0,0.0,0.7628,-0.6877,0.000,0.13190,0.000000,0.2681
97,-2.02,-0.25,329.6,13.19,14.260,24.810,20.700,0.5704,0.8282,2,...,0.0,0.0,0.0,0.0,0.0000,-0.8216,2.426,0.04336,0.000000,0.3630
98,-2.37,-0.43,418.7,16.75,14.150,0.000,19.910,0.5662,0.7964,3,...,0.0,0.0,0.0,0.0,0.7026,-0.7749,0.000,0.00000,0.036110,0.1902


In [20]:
# Load the target variable (Log_S1) for all molecular structures.
y_data = pd.read_excel('Log_s1.xlsx')

In [21]:
# Inspect the target variable table.
y_data

,Log_S1
0,2.586137
1,2.593397
2,2.571825
3,2.580697
4,2.583426
...,...
95,2.682055
96,2.564429
97,2.512017
98,2.556182


In [22]:
# Assign the same molecule identifiers to the target table
# to ensure consistent alignment with the descriptor matrix.
y_data.index = range(1, len(df) + 1)

y_data.index.name = 'SrNo'

In [23]:
# Verify that the target variable and descriptor matrix use
# consistent molecule identifiers.
y_data

,Log_S1
SrNo,
1,2.586137
2,2.593397
3,2.571825
4,2.580697
5,2.583426
...,...
96,2.682055
97,2.564429
98,2.512017


In [24]:
# Create a contiguous copy of the descriptor DataFrame.
# This avoids potential DataFrame fragmentation and improves
# performance during subsequent column operations.
df = df.copy()

In [25]:
# Add Log_S1 to the descriptor table to create a complete dataset
# containing both molecular descriptors and the target variable.
df['Log_S1'] = y_data['Log_S1'].values

In [26]:
# Verify the Log_S1 values added to the main dataset.
df['Log_S1']

SrNo
1      2.586137
2      2.593397
3      2.571825
4      2.580697
5      2.583426
         ...   
96     2.682055
97     2.564429
98     2.512017
99     2.556182
100    2.513750
Name: Log_S1, Length: 100, dtype: float64

In [28]:
# Inspect the complete preprocessed dataset before train/test separation.
df

,ALogPS_logP,ALogPS_logS,MW:(alvaDesc),AMW:(alvaDesc),Sv:(alvaDesc),Se:(alvaDesc),Sp:(alvaDesc),Mv:(alvaDesc),Mp:(alvaDesc),nTA:(alvaDesc),...,Se1Ag1Au2s:(OEstate),Se1Cu1Au2s:(OEstate),Se1Cu1Cu2s:(OEstate),Se1N3Cu2ss:(OEstate),SssCu:(OEstate),Se1O2Cu2ss:(OEstate),Se1Cu2Ag1s:(OEstate),Se1Cu2Au1s:(OEstate),Se1C3C3ss:(OEstate),Log_S1
SrNo,,,,,,,,,,,,,,,,,,,,,
1,-1.30,-0.15,215.7,107.90,2.071,1.331,8.182,1.0360,4.0910,2,...,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000,2.586137
2,-1.30,-0.43,304.8,152.40,1.967,0.000,7.386,0.9833,3.6930,2,...,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000,2.593397
3,-1.30,-0.08,171.4,85.71,1.594,1.385,7.557,0.7969,3.7780,2,...,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000,2.571825
4,-1.30,-0.61,393.9,197.00,1.862,0.000,6.591,0.9310,3.2960,2,...,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000,2.580697
5,-1.30,-0.27,260.5,130.30,1.489,0.000,6.761,0.7447,3.3810,2,...,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000,2.583426
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,-2.43,0.18,329.6,13.19,14.260,24.810,20.700,0.5704,0.8282,3,...,0.0,0.0,0.0,0.7628,-0.6877,0.000,0.13190,0.000000,0.2681,2.682055
97,-2.02,-0.25,329.6,13.19,14.260,24.810,20.700,0.5704,0.8282,2,...,0.0,0.0,0.0,0.0000,-0.8216,2.426,0.04336,0.000000,0.3630,2.564429
98,-2.37,-0.43,418.7,16.75,14.150,0.000,19.910,0.5662,0.7964,3,...,0.0,0.0,0.0,0.7026,-0.7749,0.000,0.00000,0.036110,0.1902,2.512017


In [29]:
# Save the correlation-filtered descriptor matrix together with Log_S1
# as an intermediate preprocessing output.
df.to_excel("All_molecules_noncorelated_descriptors_and_y.xlsx", index=True)

In [29]:
# Define the pool of structures eligible for external test selection.
#
# The dataset contains isolated metal clusters, individual DNA bases,
# and metal–base complexes. Because the intended prediction domain
# consists of metal–base complexes, isolated clusters and bases are
# excluded from the test candidate pool.
#
# These excluded structures are retained in the final training set.
df_train = df.drop(index=list(range(1, 8)) + [26, 45, 73]).copy()

In [30]:
# Inspect the subset containing structures eligible for external test selection.
df_train

,ALogPS_logP,ALogPS_logS,MW:(alvaDesc),AMW:(alvaDesc),Sv:(alvaDesc),Se:(alvaDesc),Sp:(alvaDesc),Mv:(alvaDesc),Mp:(alvaDesc),nTA:(alvaDesc),...,Se1Ag1Au2s:(OEstate),Se1Cu1Au2s:(OEstate),Se1Cu1Cu2s:(OEstate),Se1N3Cu2ss:(OEstate),SssCu:(OEstate),Se1O2Cu2ss:(OEstate),Se1Cu2Ag1s:(OEstate),Se1Cu2Au1s:(OEstate),Se1C3C3ss:(OEstate),Log_S1
SrNo,,,,,,,,,,,,,,,,,,,,,
8,-2.11,-0.29,331.9,16.60,11.69,19.56,18.32,0.5847,0.9159,3,...,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000,2.784332
9,-1.42,-0.60,331.9,16.60,11.69,19.56,18.32,0.5847,0.9159,2,...,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000,2.577377
10,-1.90,-0.78,421.0,21.05,11.59,0.00,17.52,0.5795,0.8762,3,...,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000,2.657629
11,-1.41,-1.05,421.0,21.05,11.59,0.00,17.52,0.5795,0.8762,2,...,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000,2.639188
12,-2.22,-0.03,287.6,14.38,11.22,19.61,17.69,0.5608,0.8847,3,...,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000,2.774590
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,-2.43,0.18,329.6,13.19,14.26,24.81,20.70,0.5704,0.8282,3,...,0.0,0.0,0.0,0.7628,-0.6877,0.000,0.13190,0.000000,0.2681,2.682055
97,-2.02,-0.25,329.6,13.19,14.26,24.81,20.70,0.5704,0.8282,2,...,0.0,0.0,0.0,0.0000,-0.8216,2.426,0.04336,0.000000,0.3630,2.564429
98,-2.37,-0.43,418.7,16.75,14.15,0.00,19.91,0.5662,0.7964,3,...,0.0,0.0,0.0,0.7026,-0.7749,0.000,0.00000,0.036110,0.1902,2.512017


In [31]:
# Prepare descriptors (X) and target values (y) only for selecting
# the external test molecules.
#
# IMPORTANT:
# These objects are used exclusively to define the test split.
# Subsequent feature selection must be performed on the training set only
# to prevent information leakage from the external test set.
X = df_train.drop(columns=['Log_S1'])
y = df_train['Log_S1']

In [32]:
# Inspect the descriptor matrix used to construct the external test split.
X

,ALogPS_logP,ALogPS_logS,MW:(alvaDesc),AMW:(alvaDesc),Sv:(alvaDesc),Se:(alvaDesc),Sp:(alvaDesc),Mv:(alvaDesc),Mp:(alvaDesc),nTA:(alvaDesc),...,Se1O2Au2ss:(OEstate),Se1Ag1Au2s:(OEstate),Se1Cu1Au2s:(OEstate),Se1Cu1Cu2s:(OEstate),Se1N3Cu2ss:(OEstate),SssCu:(OEstate),Se1O2Cu2ss:(OEstate),Se1Cu2Ag1s:(OEstate),Se1Cu2Au1s:(OEstate),Se1C3C3ss:(OEstate)
SrNo,,,,,,,,,,,,,,,,,,,,,
8,-2.11,-0.29,331.9,16.60,11.69,19.56,18.32,0.5847,0.9159,3,...,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000
9,-1.42,-0.60,331.9,16.60,11.69,19.56,18.32,0.5847,0.9159,2,...,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000
10,-1.90,-0.78,421.0,21.05,11.59,0.00,17.52,0.5795,0.8762,3,...,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000
11,-1.41,-1.05,421.0,21.05,11.59,0.00,17.52,0.5795,0.8762,2,...,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000
12,-2.22,-0.03,287.6,14.38,11.22,19.61,17.69,0.5608,0.8847,3,...,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,-2.43,0.18,329.6,13.19,14.26,24.81,20.70,0.5704,0.8282,3,...,0.0,0.0,0.0,0.0,0.7628,-0.6877,0.000,0.13190,0.000000,0.2681
97,-2.02,-0.25,329.6,13.19,14.26,24.81,20.70,0.5704,0.8282,2,...,0.0,0.0,0.0,0.0,0.0000,-0.8216,2.426,0.04336,0.000000,0.3630
98,-2.37,-0.43,418.7,16.75,14.15,0.00,19.91,0.5662,0.7964,3,...,0.0,0.0,0.0,0.0,0.7026,-0.7749,0.000,0.00000,0.036110,0.1902


In [33]:
# Inspect the target vector used to construct the external test split.
y

SrNo
8      2.784332
9      2.577377
10     2.657629
11     2.639188
12     2.774590
         ...   
96     2.682055
97     2.564429
98     2.512017
99     2.556182
100    2.513750
Name: Log_S1, Length: 90, dtype: float64

In [34]:
# Randomly select approximately 20 complexes for the external test set.
# A fixed random_state ensures that the same molecules are selected
# in every reproducible run.
#
# The remaining complexes, together with isolated clusters and DNA bases,
# are retained for model development.
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.22, random_state=42)

In [35]:
# Inspect the descriptor matrix of the selected external test complexes.
X_test

,ALogPS_logP,ALogPS_logS,MW:(alvaDesc),AMW:(alvaDesc),Sv:(alvaDesc),Se:(alvaDesc),Sp:(alvaDesc),Mv:(alvaDesc),Mp:(alvaDesc),nTA:(alvaDesc),...,Se1O2Au2ss:(OEstate),Se1Ag1Au2s:(OEstate),Se1Cu1Au2s:(OEstate),Se1Cu1Cu2s:(OEstate),Se1N3Cu2ss:(OEstate),SssCu:(OEstate),Se1O2Cu2ss:(OEstate),Se1Cu2Ag1s:(OEstate),Se1Cu2Au1s:(OEstate),Se1C3C3ss:(OEstate)
SrNo,,,,,,,,,,,,,,,,,,,,,
50,-2.20,-0.72,447.1,18.63,13.92,0.00,20.08,0.5799,0.8367,2,...,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.00000,0.00000,0.6872
31,-1.12,-0.43,302.6,13.75,12.44,21.72,18.90,0.5653,0.8593,3,...,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.00000,0.00000,0.2119
65,-2.30,0.45,269.3,11.22,13.07,23.54,19.63,0.5445,0.8177,2,...,0.000,0.0000,0.0000,0.2929,0.8456,-0.5617,0.000,0.00000,0.00000,0.6738
81,-2.43,0.18,329.6,13.19,14.26,24.81,20.70,0.5704,0.8282,3,...,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.00000,0.00000,0.2703
8,-2.11,-0.29,331.9,16.60,11.69,19.56,18.32,0.5847,0.9159,3,...,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.00000,0.00000,0.0000
35,-1.07,-1.01,436.0,19.82,12.81,0.00,18.73,0.5822,0.8515,3,...,2.253,0.0484,0.0000,0.0000,0.0000,0.0000,0.000,0.00000,0.00000,0.2140
49,-2.18,-0.75,447.1,18.63,13.92,0.00,20.08,0.5799,0.8367,2,...,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.00000,0.00000,0.6872
76,-2.00,-0.53,374.0,14.96,14.74,24.76,21.33,0.5895,0.8532,2,...,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.00000,0.00000,0.3693
18,-2.00,-0.52,376.7,18.83,11.11,0.00,16.90,0.5556,0.8449,3,...,0.000,0.0000,0.2465,0.0000,0.0000,0.0000,0.000,0.00000,0.00000,0.0000


In [36]:
# Record the molecule identifiers assigned to the external test set.
# The identifiers are saved separately to ensure that exactly the same
# external test set can be reproduced in subsequent modeling workflows.
test_molecules = X_test.index.tolist()

print("Test set molecule IDs:", test_molecules)
pd.Series(test_molecules, name="SrNo").to_csv("test_molecules.csv", index=False)

Номера молекул в тесте: [50, 31, 65, 81, 8, 35, 49, 76, 18, 54, 92, 44, 66, 97, 20, 12, 27, 37, 59, 72]


In [37]:
# Remove the selected external test molecules from the complete dataset.
#
# The resulting dataset contains all structures available for model
# development, including isolated clusters, DNA bases, and the remaining
# metal–base complexes.
df_split = df.drop(index=test_molecules).copy()

In [38]:
# Inspect the final dataset available for descriptor selection
# and model development.
df_split

,ALogPS_logP,ALogPS_logS,MW:(alvaDesc),AMW:(alvaDesc),Sv:(alvaDesc),Se:(alvaDesc),Sp:(alvaDesc),Mv:(alvaDesc),Mp:(alvaDesc),nTA:(alvaDesc),...,Se1Ag1Au2s:(OEstate),Se1Cu1Au2s:(OEstate),Se1Cu1Cu2s:(OEstate),Se1N3Cu2ss:(OEstate),SssCu:(OEstate),Se1O2Cu2ss:(OEstate),Se1Cu2Ag1s:(OEstate),Se1Cu2Au1s:(OEstate),Se1C3C3ss:(OEstate),Log_S1
SrNo,,,,,,,,,,,,,,,,,,,,,
1,-1.30,-0.15,215.7,107.90,2.071,1.331,8.182,1.0360,4.0910,2,...,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000,2.586137
2,-1.30,-0.43,304.8,152.40,1.967,0.000,7.386,0.9833,3.6930,2,...,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000,2.593397
3,-1.30,-0.08,171.4,85.71,1.594,1.385,7.557,0.7969,3.7780,2,...,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000,2.571825
4,-1.30,-0.61,393.9,197.00,1.862,0.000,6.591,0.9310,3.2960,2,...,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000,2.580697
5,-1.30,-0.27,260.5,130.30,1.489,0.000,6.761,0.7447,3.3810,2,...,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000,2.583426
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,-2.45,0.18,329.6,13.19,14.260,24.810,20.700,0.5704,0.8282,3,...,0.0,0.0,0.0,0.6965,-0.7871,0.000,0.08378,0.000000,0.1877,2.557748
96,-2.43,0.18,329.6,13.19,14.260,24.810,20.700,0.5704,0.8282,3,...,0.0,0.0,0.0,0.7628,-0.6877,0.000,0.13190,0.000000,0.2681,2.682055
98,-2.37,-0.43,418.7,16.75,14.150,0.000,19.910,0.5662,0.7964,3,...,0.0,0.0,0.0,0.7026,-0.7749,0.000,0.00000,0.036110,0.1902,2.512017


In [39]:
# Separate molecular descriptors and the Log_S1 target
# for the model-development dataset.
X_split = df_split.drop(columns=['Log_S1'])
y_split = df_split['Log_S1']

In [40]:
# Inspect the descriptor matrix used for descriptor selection.
X_split

,ALogPS_logP,ALogPS_logS,MW:(alvaDesc),AMW:(alvaDesc),Sv:(alvaDesc),Se:(alvaDesc),Sp:(alvaDesc),Mv:(alvaDesc),Mp:(alvaDesc),nTA:(alvaDesc),...,Se1O2Au2ss:(OEstate),Se1Ag1Au2s:(OEstate),Se1Cu1Au2s:(OEstate),Se1Cu1Cu2s:(OEstate),Se1N3Cu2ss:(OEstate),SssCu:(OEstate),Se1O2Cu2ss:(OEstate),Se1Cu2Ag1s:(OEstate),Se1Cu2Au1s:(OEstate),Se1C3C3ss:(OEstate)
SrNo,,,,,,,,,,,,,,,,,,,,,
1,-1.30,-0.15,215.7,107.90,2.071,1.331,8.182,1.0360,4.0910,2,...,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000
2,-1.30,-0.43,304.8,152.40,1.967,0.000,7.386,0.9833,3.6930,2,...,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000
3,-1.30,-0.08,171.4,85.71,1.594,1.385,7.557,0.7969,3.7780,2,...,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000
4,-1.30,-0.61,393.9,197.00,1.862,0.000,6.591,0.9310,3.2960,2,...,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000
5,-1.30,-0.27,260.5,130.30,1.489,0.000,6.761,0.7447,3.3810,2,...,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,-2.45,0.18,329.6,13.19,14.260,24.810,20.700,0.5704,0.8282,3,...,0.0,0.0,0.0,0.0,0.6965,-0.7871,0.000,0.08378,0.000000,0.1877
96,-2.43,0.18,329.6,13.19,14.260,24.810,20.700,0.5704,0.8282,3,...,0.0,0.0,0.0,0.0,0.7628,-0.6877,0.000,0.13190,0.000000,0.2681
98,-2.37,-0.43,418.7,16.75,14.150,0.000,19.910,0.5662,0.7964,3,...,0.0,0.0,0.0,0.0,0.7026,-0.7749,0.000,0.00000,0.036110,0.1902


In [41]:
# Recombine the training descriptors and target variable into a single table
# required for preparing GA-MLR input files.
#
# Molecules are sorted by their original identifiers to preserve the
# correspondence with the source dataset.
train_chunk = X_split.copy()
train_chunk['Log_S1'] = y_split
train_chunk = train_chunk.sort_index()

In [42]:
# Inspect the training table before dividing descriptors into GA input chunks.
train_chunk

,ALogPS_logP,ALogPS_logS,MW:(alvaDesc),AMW:(alvaDesc),Sv:(alvaDesc),Se:(alvaDesc),Sp:(alvaDesc),Mv:(alvaDesc),Mp:(alvaDesc),nTA:(alvaDesc),...,Se1Ag1Au2s:(OEstate),Se1Cu1Au2s:(OEstate),Se1Cu1Cu2s:(OEstate),Se1N3Cu2ss:(OEstate),SssCu:(OEstate),Se1O2Cu2ss:(OEstate),Se1Cu2Ag1s:(OEstate),Se1Cu2Au1s:(OEstate),Se1C3C3ss:(OEstate),Log_S1
SrNo,,,,,,,,,,,,,,,,,,,,,
1,-1.30,-0.15,215.7,107.90,2.071,1.331,8.182,1.0360,4.0910,2,...,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000,2.586137
2,-1.30,-0.43,304.8,152.40,1.967,0.000,7.386,0.9833,3.6930,2,...,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000,2.593397
3,-1.30,-0.08,171.4,85.71,1.594,1.385,7.557,0.7969,3.7780,2,...,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000,2.571825
4,-1.30,-0.61,393.9,197.00,1.862,0.000,6.591,0.9310,3.2960,2,...,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000,2.580697
5,-1.30,-0.27,260.5,130.30,1.489,0.000,6.761,0.7447,3.3810,2,...,0.0,0.0,0.0,0.0000,0.0000,0.000,0.00000,0.000000,0.0000,2.583426
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,-2.45,0.18,329.6,13.19,14.260,24.810,20.700,0.5704,0.8282,3,...,0.0,0.0,0.0,0.6965,-0.7871,0.000,0.08378,0.000000,0.1877,2.557748
96,-2.43,0.18,329.6,13.19,14.260,24.810,20.700,0.5704,0.8282,3,...,0.0,0.0,0.0,0.7628,-0.6877,0.000,0.13190,0.000000,0.2681,2.682055
98,-2.37,-0.43,418.7,16.75,14.150,0.000,19.910,0.5662,0.7964,3,...,0.0,0.0,0.0,0.7026,-0.7749,0.000,0.00000,0.036110,0.1902,2.512017


In [43]:
# -------------------------------------------------------------------------
# Prepare descriptor chunks for GA-MLR feature selection
# -------------------------------------------------------------------------
#
# The training descriptor matrix contains substantially more descriptors
# than can be processed conveniently in a single GA-MLR run.
# Therefore, descriptors are divided into non-overlapping chunks of
# 100 features.
#
# Each output Excel file contains:
#   1. molecule identifier ("SR No");
#   2. up to 100 molecular descriptors;
#   3. target variable (Log_S1).
#
# The resulting files are used as independent inputs for GA-MLR analysis
# in DTC Lab Tools. External test molecules are not included at this stage.
# -------------------------------------------------------------------------
import os

# Create the output folder if it does not exist
output_folder = 'Chunk_for_GA'
os.makedirs(output_folder, exist_ok=True)

chunk_size = 100

# train_chunk: index = SrNo, columns = descriptors + Log_S1
X = train_chunk.drop(columns=['Log_S1'])
y = train_chunk['Log_S1']

# Split descriptors into chunks
for i in range(0, X.shape[1], chunk_size):
    chunk = X.iloc[:, i:i + chunk_size].copy()

    # Insert index as the first column (required for DTC Lab Tools GA)
    chunk.insert(0, 'SR No', chunk.index)

    # Add the target variable as the last column
    chunk['Log_S1'] = y.values

    # Save the file
    file_name = f'GA_chunk_DNA_S1_{i + 1}-{i + chunk.shape[1]}.xlsx'
    chunk.to_excel(os.path.join(output_folder, file_name), index=False)



In [49]:
# -------------------------------------------------------------------------
# Consensus descriptor selection from GA-MLR results
# -------------------------------------------------------------------------
#
# GA-MLR output files contain multiple candidate regression equations.
# Descriptor robustness is assessed from its frequency of occurrence
# across these independently generated equations.
#
# For each GA output file:
#   1. regression equations beginning with "Log_S1 =" are identified;
#   2. descriptors present in each equation are extracted;
#   3. the fraction of equations containing each descriptor is calculated;
#   4. descriptors with selection frequency >= min_freq are retained;
#   5. at most top_k highest-frequency descriptors are selected.
#
# Selected descriptors from all descriptor chunks are then combined
# into a single consensus feature pool.
#
# This procedure favors descriptors that are repeatedly selected by GA-MLR
# rather than descriptors appearing only in isolated candidate models.
# -------------------------------------------------------------------------
import os
import re
from collections import Counter

folder_path = "GA_answer"
top_k = 10
min_freq = 0.4  # minimum frequency threshold (e.g., 0.3–0.4 to filter rare descriptors)

# Regex pattern to capture descriptors of the form Name:(Source)
desc_pattern = re.compile(r"([^\s]+:\([^)]+\))")

txt_files = sorted([f for f in os.listdir(folder_path) if f.endswith(".txt")])

final_pool = set()

for filename in txt_files:
    file_path = os.path.join(folder_path, filename)

    descriptor_model_count = Counter()
    total_models = 0

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            # Only parse regression equations
            if not line.startswith("Log_S1 ="):
                continue

            total_models += 1
            # Extract descriptors appearing in the equation
            descs = set(desc_pattern.findall(line))  # each model counts as one vote
            for d in descs:
                descriptor_model_count[d] += 1

    if total_models == 0:
        print(f"[WARN] No equations found in {filename}")
        continue

    # Calculate descriptor frequencies across GA models
    freqs = {d: c / total_models for d, c in descriptor_model_count.items()}
    # Rank descriptors by frequency
    ranked = sorted(freqs.items(), key=lambda x: x[1], reverse=True)

    # Apply frequency threshold and select top-K descriptors
    top_descs = [d for d, fr in ranked if fr >= min_freq][:top_k]
    final_pool.update(top_descs)

    print("\n" + "=" * 60)
    print(f"File: {filename}")
    print(f"Total GA models: {total_models}")
    print(f"Selected (top {top_k}, min_freq={min_freq}): {len(top_descs)}\n")
    for d in top_descs:
        print(f"{d}: {freqs[d]:.2f}")

print("\n" + "#" * 60)
print(f"FINAL POOL size = {len(final_pool)}\n")
for d in sorted(final_pool):
    print(d)


File: DNA_S1_1_GA(MLR)model.txt
Total GA models: 10
Selected (top 10, min_freq=0.4): 7

J_Dz(p):(alvaDesc): 1.00
ChiA_X:(alvaDesc): 0.90
VE3sign_Dz(e):(alvaDesc): 0.70
ZM1V:(alvaDesc): 0.50
nHM:(alvaDesc): 0.50
VE1sign_D/Dt:(alvaDesc): 0.40
ZM1Mad:(alvaDesc): 0.40

File: DNA_S1_2_GA(MLR)model.txt
Total GA models: 13
Selected (top 10, min_freq=0.4): 4

JGI2:(alvaDesc): 0.69
GATS1i:(alvaDesc): 0.69
MATS3i:(alvaDesc): 0.62
MATS1i:(alvaDesc): 0.54

File: DNA_S1_3_GA(MLR)model.txt
Total GA models: 12
Selected (top 10, min_freq=0.4): 7

VE1sign_G:(alvaDesc): 0.83
SpMin8_Bh(m):(alvaDesc): 0.67
DISPp:(alvaDesc): 0.58
P_VSA_s_5:(alvaDesc): 0.58
DISPm:(alvaDesc): 0.50
ASP:(alvaDesc): 0.42
QXXs:(alvaDesc): 0.42

File: DNA_S1_4_GA(MLR)model.txt
Total GA models: 11
Selected (top 10, min_freq=0.4): 10

TDB07p:(alvaDesc): 0.73
RDF060u:(alvaDesc): 0.64
RDF055p:(alvaDesc): 0.64
RDF055v:(alvaDesc): 0.45
Mor06u:(alvaDesc): 0.45
RDF060e:(alvaDesc): 0.45
RDF020p:(alvaDesc): 0.45
RDF045s:(alvaDesc): 0.45
M

In [50]:
# Save the consensus GA-selected descriptor pool.
# Sorting the descriptor names ensures deterministic and reproducible output.
output_file = os.path.join(folder_path, "FINAL_top10_04_pool.txt")

with open(output_file, "w", encoding="utf-8") as f:
    for desc in sorted(final_pool):
        f.write(desc + "\n")

print(f"\nFinal pool saved to: {output_file}")


Final pool saved to: GA_answer\FINAL_top10_04_pool.txt


In [51]:
# Load the final consensus descriptor list generated by the GA-MLR
# frequency analysis.
with open("GA_answer/FINAL_top10_04_pool.txt", "r", encoding="utf-8") as f:
    selected_descriptors = [line.strip() for line in f]

print("Number of selected descriptors:", len(selected_descriptors))

Number of selected descriptors: 62


In [52]:
# Verify that all GA-selected descriptors are present in the original
# training descriptor matrix.
#
# Missing descriptors are reported explicitly, while available descriptors
# are extracted to construct the final modeling dataset.
existing = [d for d in selected_descriptors if d in train_chunk.columns]
missing = [d for d in selected_descriptors if d not in train_chunk.columns]

print("Found:", len(existing))
print("Missing:", len(missing))

# Create a new table containing only the descriptors that exist in the dataset
clean_descriptors = train_chunk[existing].copy()

print("New table shape:", clean_descriptors.shape)

Found: 62
Missing: 0
New table shape: (80, 62)


In [53]:
# Inspect the descriptor matrix after consensus GA selection.
clean_descriptors

,ASP:(alvaDesc),ChiA_X:(alvaDesc),DISPm:(alvaDesc),DISPp:(alvaDesc),DLS_cons:(alvaDesc),Dp:(alvaDesc),Ds:(alvaDesc),E2u:(alvaDesc),F01[C-C]:(alvaDesc),F03[C-X]:(alvaDesc),...,SsCu:(OEstate),TDB07p:(alvaDesc),TDB07s:(alvaDesc),VE1sign_D/Dt:(alvaDesc),VE1sign_G:(alvaDesc),VE3sign_Dz(e):(alvaDesc),ZM1Mad:(alvaDesc),ZM1V:(alvaDesc),nHM:(alvaDesc),s4_numRotBonds:(alvaDesc)
SrNo,,,,,,,,,,,,,,,,,,,,,
1,0.0000,1.0000,0.00,0.000,0.7952,0.0000,0.0000,0.0000,0,0,...,0.000,0.000,0.000,0.00000,0.00000,0.0000,161.3,242,2,0.00
2,0.0000,1.0000,0.00,0.000,0.7952,0.0000,0.0000,0.0000,0,0,...,0.000,0.000,0.000,0.00000,0.00000,0.0000,349.6,242,2,0.00
3,0.0000,1.0000,0.00,0.000,0.7476,0.0000,0.0000,0.0000,0,0,...,1.273,0.000,0.000,0.00000,0.00000,-0.3430,108.7,242,2,0.00
4,0.0000,1.0000,0.00,0.000,0.7952,0.0000,0.0000,0.0000,0,0,...,0.000,0.000,0.000,0.00000,0.00000,0.0000,538.0,242,2,0.00
5,0.0000,1.0000,0.00,0.000,0.7952,0.0000,0.0000,0.0000,0,0,...,1.285,0.000,0.000,0.00000,0.00000,0.0000,297.0,242,2,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.3837,0.9944,21.10,1.845,0.8643,0.6952,0.4203,0.4966,2,3,...,0.000,1.729,6.909,0.15100,0.03175,-0.5103,249.2,389,2,1.00
96,0.2660,0.9953,22.42,2.062,0.8643,0.8535,0.4053,0.4266,2,4,...,0.000,6.350,18.860,0.08457,0.15040,-0.6110,249.2,389,2,0.75
98,0.5437,0.9944,27.40,1.632,0.8643,0.6262,0.3892,0.4966,2,3,...,0.000,1.538,6.571,0.15100,0.03175,0.0000,454.8,389,2,1.00


In [54]:
# Add the Log_S1 target variable to the final selected descriptor matrix.
clean_descriptors["Log_S1"] = train_chunk["Log_S1"]

In [55]:
# Inspect the final GA-selected modeling dataset.
clean_descriptors

,ASP:(alvaDesc),ChiA_X:(alvaDesc),DISPm:(alvaDesc),DISPp:(alvaDesc),DLS_cons:(alvaDesc),Dp:(alvaDesc),Ds:(alvaDesc),E2u:(alvaDesc),F01[C-C]:(alvaDesc),F03[C-X]:(alvaDesc),...,TDB07p:(alvaDesc),TDB07s:(alvaDesc),VE1sign_D/Dt:(alvaDesc),VE1sign_G:(alvaDesc),VE3sign_Dz(e):(alvaDesc),ZM1Mad:(alvaDesc),ZM1V:(alvaDesc),nHM:(alvaDesc),s4_numRotBonds:(alvaDesc),Log_S1
SrNo,,,,,,,,,,,,,,,,,,,,,
1,0.0000,1.0000,0.00,0.000,0.7952,0.0000,0.0000,0.0000,0,0,...,0.000,0.000,0.00000,0.00000,0.0000,161.3,242,2,0.00,2.586137
2,0.0000,1.0000,0.00,0.000,0.7952,0.0000,0.0000,0.0000,0,0,...,0.000,0.000,0.00000,0.00000,0.0000,349.6,242,2,0.00,2.593397
3,0.0000,1.0000,0.00,0.000,0.7476,0.0000,0.0000,0.0000,0,0,...,0.000,0.000,0.00000,0.00000,-0.3430,108.7,242,2,0.00,2.571825
4,0.0000,1.0000,0.00,0.000,0.7952,0.0000,0.0000,0.0000,0,0,...,0.000,0.000,0.00000,0.00000,0.0000,538.0,242,2,0.00,2.580697
5,0.0000,1.0000,0.00,0.000,0.7952,0.0000,0.0000,0.0000,0,0,...,0.000,0.000,0.00000,0.00000,0.0000,297.0,242,2,0.00,2.583426
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.3837,0.9944,21.10,1.845,0.8643,0.6952,0.4203,0.4966,2,3,...,1.729,6.909,0.15100,0.03175,-0.5103,249.2,389,2,1.00,2.557748
96,0.2660,0.9953,22.42,2.062,0.8643,0.8535,0.4053,0.4266,2,4,...,6.350,18.860,0.08457,0.15040,-0.6110,249.2,389,2,0.75,2.682055
98,0.5437,0.9944,27.40,1.632,0.8643,0.6262,0.3892,0.4966,2,3,...,1.538,6.571,0.15100,0.03175,0.0000,454.8,389,2,1.00,2.512017


In [56]:
# Save the final consensus descriptor dataset for downstream QSPR modeling.
#
# This descriptor pool is used as the starting feature set for subsequent
# machine-learning models. MLR is treated separately and undergoes an
# additional GA-based reduction to a compact 10-descriptor model.
clean_descriptors.to_excel("clean_descriptors_62.xlsx", index=True)